# MIA Summary Explorer
Interactively group and compare rows in results/summary_by_group.csv.

- Group on the same (dataset, sparsity, attack, metric)
- Compare across (method, mode) with stats: mean, std, CI, n (unique victims)


In [2]:
# Quick MPS check for macOS (PyTorch)
try:
    import torch
    if torch.backends.mps.is_available():
        mps_device = torch.device('mps')
        x = torch.ones(1, device=mps_device)
        print(x)
    else:
        print('MPS device not found.')
except Exception as e:
    print('PyTorch not available or MPS check failed:', e)


tensor([1.], device='mps:0')


In [19]:
import pandas as pd
from pathlib import Path
from IPython.display import display

CSV_PATH = Path('../results/summary_by_group.csv')
assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'
df = pd.read_csv(CSV_PATH)
if 'sparsity' in df.columns:
    df['sparsity'] = pd.to_numeric(df['sparsity'], errors='coerce')
if 'mode' not in df.columns:
    df['mode'] = ''

tidy_path = Path('../results/results_raw_tidy.csv')
assert tidy_path.exists(), f'Tidy CSV not found: {tidy_path}'
df_seed = pd.read_csv(tidy_path)
for col in ('sparsity', 'victim_seed', 'value'):
    if col in df_seed.columns:
        df_seed[col] = pd.to_numeric(df_seed[col], errors='coerce')
if 'victim_seed' in df_seed.columns:
    df_seed['victim_seed'] = df_seed['victim_seed'].round().astype('Int64')
for col in ('dataset', 'method', 'mode', 'attack', 'metric'):
    if col in df_seed.columns:
        df_seed[col] = df_seed[col].fillna('').astype(str)

display(df_seed.head())


,attack,metric,value,file,dataset,arch,method,mode,forward_mode,sparsity,victim_seed,victim_test_acc,use_temperature,attack_mode,alpha,beta
0,lira,auroc,0.542107,mia_results/dwa_kill_active_plain_dead/sparsit...,cifar10,NaN,dwa,kill_active_plain_dead,dwa_adaptive,0.9,46,89.0,0,pruned,5.0,5.0
1,lira,ap,0.980449,mia_results/dwa_kill_active_plain_dead/sparsit...,cifar10,NaN,dwa,kill_active_plain_dead,dwa_adaptive,0.9,46,89.0,0,pruned,5.0,5.0
2,lira,advantage,0.054356,mia_results/dwa_kill_active_plain_dead/sparsit...,cifar10,NaN,dwa,kill_active_plain_dead,dwa_adaptive,0.9,46,89.0,0,pruned,5.0,5.0
3,lira,accuracy,0.920478,mia_results/dwa_kill_active_plain_dead/sparsit...,cifar10,NaN,dwa,kill_active_plain_dead,dwa_adaptive,0.9,46,89.0,0,pruned,5.0,5.0
4,lira,balanced_accuracy,0.527178,mia_results/dwa_kill_active_plain_dead/sparsit...,cifar10,NaN,dwa,kill_active_plain_dead,dwa_adaptive,0.9,46,89.0,0,pruned,5.0,5.0


In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

def pivot_block(dataset, sparsity, attack, metric, sort_by='mean', ascending=False):
    sub = df.copy()
    if dataset is not None:
        sub = sub[sub['dataset'] == dataset]
    sub = sub[(sub['sparsity'] == sparsity) & (sub['attack'] == attack) & (sub['metric'] == metric)]
    if sub.empty:
        return pd.DataFrame(), pd.DataFrame()
    cols = ['mean','std','ci95_lo','ci95_hi','n','median','iqr']
    show = ['dataset','sparsity','attack','metric','method','mode'] + cols
    sub = sub[show].copy()
    pvt = (sub.set_index(['method','mode'])[cols]
              .sort_values(by=[sort_by], ascending=ascending))
    return pvt, sub.sort_values(by=[sort_by], ascending=ascending)

datasets   = sorted(df['dataset'].dropna().unique().tolist())
attacks    = sorted(df['attack'].dropna().unique().tolist())
metrics    = sorted(df['metric'].dropna().unique().tolist())
sparsities = sorted(df['sparsity'].dropna().unique().tolist())

dd_dataset = W.Dropdown(options=['(all)'] + datasets, value=(datasets[0] if datasets else None), description='dataset')
dd_attack  = W.Dropdown(options=attacks, value=(attacks[0] if attacks else None), description='attack')
dd_metric  = W.Dropdown(options=metrics, value=(metrics[0] if metrics else None), description='metric')
dd_spars   = W.Dropdown(options=sparsities, value=(sparsities[0] if sparsities else None), description='sparsity')
dd_sort    = W.Dropdown(options=['mean','median','n','std','ci95_lo','ci95_hi','iqr'], value='mean', description='sort by')
dd_asc     = W.Checkbox(value=False, description='ascending')

out = W.Output()

def refresh(*_):
    with out:
        out.clear_output()
        ds = None if dd_dataset.value == '(all)' else dd_dataset.value
        pvt, tbl = pivot_block(ds, dd_spars.value, dd_attack.value, dd_metric.value, dd_sort.value, dd_asc.value)
        if pvt.empty:
            display(HTML('<b>No rows match the current filters.</b>'))
            return
        display(HTML('<h3>Pivot: rows = (method, mode)</h3>'))
        display(pvt)
        # display(HTML('<h4>Grouped rows</h4>'))
        # display(tbl)

for w in [dd_dataset, dd_attack, dd_metric, dd_spars, dd_sort, dd_asc]:
    w.observe(refresh, names='value')

display(W.HBox([dd_dataset, dd_attack, dd_metric]))
display(W.HBox([dd_spars, dd_sort, dd_asc]))
display(out)
refresh()


Output()

In [18]:
# seed 단위 상세 테이블을 위한 위젯 구성

def _string_options(series, allow_all=True):
    if series is None:
        return ['(all)'] if allow_all else []
    vals = sorted({str(v) for v in series if str(v).strip() and str(v) != 'nan'})
    return (['(all)'] if allow_all else []) + vals

seed_dataset_opts = _string_options(df_seed['dataset'] if 'dataset' in df_seed.columns else None)
seed_method_opts = _string_options(df_seed['method'] if 'method' in df_seed.columns else None)
seed_mode_opts = _string_options(df_seed['mode'] if 'mode' in df_seed.columns else None)
seed_attack_opts = _string_options(df_seed['attack'] if 'attack' in df_seed.columns else None)
seed_metric_opts = _string_options(df_seed['metric'] if 'metric' in df_seed.columns else None)
seed_sparsity_opts = sorted(float(v) for v in df_seed['sparsity'].dropna().unique()) if 'sparsity' in df_seed.columns else []
seed_victim_opts = sorted(int(v) for v in df_seed['victim_seed'].dropna().unique()) if 'victim_seed' in df_seed.columns else []

if not seed_attack_opts:
    seed_attack_opts = ['(all)']
if not seed_metric_opts:
    seed_metric_opts = ['(all)']
if not seed_sparsity_opts:
    seed_sparsity_opts = [0.0]
if not seed_victim_opts:
    seed_victim_opts = []

_dd_dataset = W.Dropdown(options=seed_dataset_opts or ['(all)'], description='dataset')
_dd_method = W.Dropdown(options=seed_method_opts or ['(all)'], description='method')
_dd_mode = W.Dropdown(options=seed_mode_opts or ['(all)'], description='mode')
_dd_attack = W.Dropdown(options=seed_attack_opts, description='attack', value=seed_attack_opts[0])
_dd_metric = W.Dropdown(options=seed_metric_opts, description='metric', value=seed_metric_opts[0])
_dd_spars = W.Dropdown(options=seed_sparsity_opts, description='sparsity', value=seed_sparsity_opts[0])
_seed_options = ['(all)'] + seed_victim_opts if seed_victim_opts else ['(all)']
_dd_seed = W.Dropdown(options=_seed_options, description='seed', value=_seed_options[0])
_dd_order = W.ToggleButtons(options=[('descending', False), ('ascending', True)], description='order', value=False)

_seed_out = W.Output()

def _render_seed_table(*_):
    with _seed_out:
        _seed_out.clear_output()
        sub = df_seed.copy()
        if _dd_dataset.value != '(all)':
            sub = sub[sub['dataset'] == _dd_dataset.value]
        if _dd_method.value != '(all)':
            sub = sub[sub['method'] == _dd_method.value]
        if _dd_mode.value != '(all)':
            sub = sub[sub['mode'] == _dd_mode.value]
        if _dd_attack.value != '(all)':
            sub = sub[sub['attack'] == _dd_attack.value]
        if _dd_metric.value != '(all)':
            sub = sub[sub['metric'] == _dd_metric.value]
        if _dd_seed.value != '(all)':
            sub = sub[sub['victim_seed'] == int(_dd_seed.value)]
        sub = sub[sub['sparsity'] == _dd_spars.value]
        if sub.empty:
            display(HTML('<em>조건에 맞는 결과가 없습니다.</em>'))
            return
        cols = ['victim_seed', 'sparsity', 'attack', 'metric', 'value', 'dataset', 'method', 'mode']
        existing = [c for c in cols if c in sub.columns]
        sort_order_columns = []
        ascending = []
        if 'value' in existing:
            sort_order_columns.append('value')
            ascending.append(_dd_order.value)
        for c in ['victim_seed', 'sparsity', 'attack', 'metric']:
            if c in existing and c not in sort_order_columns:
                sort_order_columns.append(c)
                ascending.append(True)
        if not sort_order_columns:
            sort_order_columns = existing
            ascending = True
        display(sub[existing].sort_values(sort_order_columns, ascending=ascending).reset_index(drop=True))

for widget in (_dd_dataset, _dd_method, _dd_mode, _dd_attack, _dd_metric, _dd_spars, _dd_seed, _dd_order):
    widget.observe(_render_seed_table, names='value')

_render_seed_table()
display(W.VBox([
    W.HBox([_dd_dataset, _dd_method, _dd_mode]),
    W.HBox([_dd_attack, _dd_metric, _dd_spars, _dd_seed, _dd_order]),
    _seed_out
]))


In [5]:
# Optional: export current pivot to CSV (run after setting widgets)
from datetime import datetime
def export_current():
    ds = None if dd_dataset.value == '(all)' else dd_dataset.value
    pvt, tbl = pivot_block(ds, dd_spars.value, dd_attack.value, dd_metric.value, dd_sort.value, dd_asc.value)
    if pvt.empty:
        print('Nothing to export'); return
    out_dir = Path('results/pivots'); out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = f"pivot_{ds or 'all'}_s{dd_spars.value}_{dd_attack.value}_{dd_metric.value}_{dd_sort.value}{'_asc' if dd_asc.value else '_desc'}_{ts}"
    pvt.to_csv(out_dir / f"{base}.csv")
    print('saved:', out_dir / f"{base}.csv")

# export_current()  # call to save current pivot
